# Learning Objectives
In this notebook, you will learn Spark Dataframe APIs.

# Question List

Solve the following questions using Spark Dataframe APIs


In [0]:
bookings = spark.table("practice.bronze_bookings")
members = spark.table("practice.bronze_members")
facilities = spark.table("practice.bronze_facilities")

### Join
1. https://pgexercises.com/questions/joins/simplejoin.html
2. https://pgexercises.com/questions/joins/simplejoin2.html
3. https://pgexercises.com/questions/joins/self2.html
4. https://pgexercises.com/questions/joins/threejoin.html (three join)
5. https://pgexercises.com/questions/joins/sub.html (subquery and join)

In [0]:
from pyspark.sql.functions import *

In [0]:

#1. How can you produce a list of the start times for bookings by members named 'David Farrell'?
result = (
    bookings.alias("b")
    .join(members.alias("m"), col("b.memid") == col("m.memid"))
    .filter((col("m.firstname") == "David") & (col("m.surname") == "Farrell"))
    .select(col("b.starttime"))
)

result.show()    


+-------------------+
|          starttime|
+-------------------+
|2012-09-18 09:00:00|
|2012-09-18 17:30:00|
|2012-09-18 13:30:00|
|2012-09-18 20:00:00|
|2012-09-19 09:30:00|
|2012-09-19 15:00:00|
|2012-09-19 12:00:00|
|2012-09-20 15:30:00|
|2012-09-20 11:30:00|
|2012-09-20 14:00:00|
|2012-09-21 10:30:00|
|2012-09-21 14:00:00|
|2012-09-22 08:30:00|
|2012-09-22 17:00:00|
|2012-09-23 08:30:00|
|2012-09-23 17:30:00|
|2012-09-23 19:00:00|
|2012-09-24 08:00:00|
|2012-09-24 16:30:00|
|2012-09-24 12:30:00|
+-------------------+
only showing top 20 rows


In [0]:
# 2. How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.
result = (
   bookings.alias("b")
   .join(facilities.alias("f"), col("b.facid") == col("f.facid"))
   .filter((col("f.name").like("Tennis Court%")) & ((col("b.starttime") >= "2012-09-21") & (col("b.starttime") < "2012-09-22")))
   .select(col("b.starttime"), col("f.name"))
   .orderBy(col("b.starttime"))
)

result.show()

+-------------------+--------------+
|          starttime|          name|
+-------------------+--------------+
|2012-09-21 08:00:00|Tennis Court 2|
|2012-09-21 08:00:00|Tennis Court 1|
|2012-09-21 09:30:00|Tennis Court 1|
|2012-09-21 10:00:00|Tennis Court 2|
|2012-09-21 11:30:00|Tennis Court 2|
|2012-09-21 12:00:00|Tennis Court 1|
|2012-09-21 13:30:00|Tennis Court 1|
|2012-09-21 14:00:00|Tennis Court 2|
|2012-09-21 15:30:00|Tennis Court 1|
|2012-09-21 16:00:00|Tennis Court 2|
|2012-09-21 17:00:00|Tennis Court 1|
|2012-09-21 18:00:00|Tennis Court 2|
+-------------------+--------------+



In [0]:
# 3. How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname).
result = (
    members.alias("m")
    .join(members.alias("r"), col("m.recommendedby") == col("r.memid"), "left")
    .select(
        col("m.firstname"), 
        col("m.surname"), 
        concat_ws(" ", col("r.firstname"), col("r.surname")).alias("recommended_by")
        )
    .orderBy(col("m.surname"), col("m.firstname"))

)

result.show()

+---------+---------+-----------------+
|firstname|  surname|   recommended_by|
+---------+---------+-----------------+
| Florence|    Bader|  Ponder Stibbons|
|     Anne|    Baker|  Ponder Stibbons|
|  Timothy|    Baker|   Jemima Farrell|
|      Tim|   Boothe|       Tim Rownam|
|   Gerald|  Butters|     Darren Smith|
|     Joan|   Coplin|    Timothy Baker|
|    Erica|  Crumpet|      Tracy Smith|
|    Nancy|     Dare|  Janice Joplette|
|    David|  Farrell|                 |
|   Jemima|  Farrell|                 |
|    GUEST|    GUEST|                 |
|  Matthew|  Genting|   Gerald Butters|
|     John|     Hunt|Millicent Purview|
|    David|    Jones|  Janice Joplette|
|  Douglas|    Jones|      David Jones|
|   Janice| Joplette|     Darren Smith|
|     Anna|Mackenzie|     Darren Smith|
|  Charles|     Owen|     Darren Smith|
|    David|   Pinker|   Jemima Farrell|
|Millicent|  Purview|      Tracy Smith|
+---------+---------+-----------------+
only showing top 20 rows


In [0]:
# 4. How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.
result = (
    bookings.alias("b")
    .join(facilities.alias("f"), col("b.facid") == col("f.facid"))
    .join(members.alias("m"), col("b.memid") == col("m.memid"))
    .filter(col("f.name").like("Tennis Court%"))
    .select(concat_ws(" ", col("m.firstname"), col("m.surname")).alias("member"), col("f.name").alias("facility"))
    .distinct()
    .orderBy(col("member"), col("facility"))
)
result.show()

+--------------+--------------+
|        member|      facility|
+--------------+--------------+
|    Anne Baker|Tennis Court 1|
|    Anne Baker|Tennis Court 2|
|  Burton Tracy|Tennis Court 1|
|  Burton Tracy|Tennis Court 2|
|  Charles Owen|Tennis Court 1|
|  Charles Owen|Tennis Court 2|
|  Darren Smith|Tennis Court 2|
| David Farrell|Tennis Court 1|
| David Farrell|Tennis Court 2|
|   David Jones|Tennis Court 1|
|   David Jones|Tennis Court 2|
|  David Pinker|Tennis Court 1|
| Douglas Jones|Tennis Court 1|
| Erica Crumpet|Tennis Court 1|
|Florence Bader|Tennis Court 1|
|Florence Bader|Tennis Court 2|
|   GUEST GUEST|Tennis Court 1|
|   GUEST GUEST|Tennis Court 2|
|Gerald Butters|Tennis Court 1|
|Gerald Butters|Tennis Court 2|
+--------------+--------------+
only showing top 20 rows


In [0]:
# # 5. How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered. (subquery and join)
recommenders = (
    members
    .select(
        col("memid").alias("rec_id"), 
        concat_ws(" ", col("firstname"), col("surname")).alias("recommender")
        )
)

result = (
    members.alias("m")
    .join(
        recommenders.alias("r"),
        col("r.rec_id") == col("m.recommendedby"),
        "left"
    )
    .select(
        concat_ws(" ", col("m.firstname"), col("m.surname")).alias("member"), 
        col("recommender")
        )
    .distinct()
    .orderBy(col("member"))
)

result.show()

# from pyspark.sql.functions import concat_ws, col

# result = (
#     members.alias("m")
#     .join(
#         members.alias("r"),
#         col("m.recommendedby") == col("r.memid"),
#         "left"
#     )
#     .select(
#         concat_ws(" ", col("m.firstname"), col("m.surname")).alias("member"),
#         concat_ws(" ", col("r.firstname"), col("r.surname")).alias("recommender")
#     )
#     .distinct()
#     .orderBy("member")
# )

# result.show()


+--------------------+---------------+
|              member|    recommender|
+--------------------+---------------+
|      Anna Mackenzie|   Darren Smith|
|          Anne Baker|Ponder Stibbons|
|        Burton Tracy|           NULL|
|        Charles Owen|   Darren Smith|
|        Darren Smith|           NULL|
|       David Farrell|           NULL|
|         David Jones|Janice Joplette|
|        David Pinker| Jemima Farrell|
|       Douglas Jones|    David Jones|
|       Erica Crumpet|    Tracy Smith|
|      Florence Bader|Ponder Stibbons|
|         GUEST GUEST|           NULL|
|      Gerald Butters|   Darren Smith|
|    Henrietta Rumney|Matthew Genting|
|Henry Worthington...|    Tracy Smith|
| Hyacinth Tupperware|           NULL|
|          Jack Smith|   Darren Smith|
|     Janice Joplette|   Darren Smith|
|      Jemima Farrell|           NULL|
|         Joan Coplin|  Timothy Baker|
+--------------------+---------------+
only showing top 20 rows


### Aggregation

1. https://pgexercises.com/questions/aggregates/count3.html Group by order by
2. https://pgexercises.com/questions/aggregates/fachours.html group by order by
3. https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition
4. https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
5. https://pgexercises.com/questions/aggregates/members1.html count distinct
6. https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join

In [0]:
# 1. Produce a count of the number of recommendations each member has made. Order by member ID.Group by order by
result = (
    members
    .groupby("recommendedby")
    .count().alias("num of recommendations")
    .filter(col("recommendedby").isNotNull())
    .withColumnRenamed("recommendedby", "member id")
    .orderBy("recommendedby")
)
result.show()

+---------+-----+
|member id|count|
+---------+-----+
|        1|    5|
|        2|    3|
|        3|    1|
|        4|    2|
|        5|    1|
|        6|    1|
|        9|    2|
|       11|    1|
|       13|    2|
|       15|    1|
|       16|    1|
|       20|    1|
|       30|    1|
+---------+-----+



In [0]:
# 2. Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.
result = (
    bookings
    .groupBy("facid")
    .agg(sum("slots").alias("slots"))
    .orderBy("facid")
)
result.show()

+-----+-----+
|facid|slots|
+-----+-----+
|    0| 1320|
|    1| 1278|
|    2| 1209|
|    3|  830|
|    4| 1404|
|    5|  228|
|    6| 1104|
|    7|  908|
|    8|  911|
+-----+-----+



In [0]:
# 3. Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.
result = (
    bookings
    .filter((col("starttime") >= "2012-09-01") & (col("starttime") < "2012-10-01"))
    .groupBy("facid")
    .agg(sum("slots").alias("slots"))
    .orderBy("slots")
)
result.show()

+-----+-----+
|facid|slots|
+-----+-----+
|    5|  122|
|    3|  422|
|    7|  426|
|    8|  471|
|    6|  540|
|    2|  570|
|    1|  588|
|    0|  591|
|    4|  648|
+-----+-----+



In [0]:
# 4. Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month.
result = (
    bookings
    .filter((col("starttime") >= "2012-01-01") & (col("starttime") < "2013-01-01"))
    .groupBy("facid", month(col("starttime")).alias("month"))
    .agg(sum("slots").alias("slots"))
    .orderBy("facid", "month")
)
result.show()

+-----+-----+-----+
|facid|month|slots|
+-----+-----+-----+
|    0|    7|  270|
|    0|    8|  459|
|    0|    9|  591|
|    1|    7|  207|
|    1|    8|  483|
|    1|    9|  588|
|    2|    7|  180|
|    2|    8|  459|
|    2|    9|  570|
|    3|    7|  104|
|    3|    8|  304|
|    3|    9|  422|
|    4|    7|  264|
|    4|    8|  492|
|    4|    9|  648|
|    5|    7|   24|
|    5|    8|   82|
|    5|    9|  122|
|    6|    7|  164|
|    6|    8|  400|
+-----+-----+-----+
only showing top 20 rows


In [0]:
# 5. Find the total number of members (including guests) who have made at least one booking. count distinct
result = (
    bookings.agg(countDistinct("memid").alias("total members"))
)
result.show()

+-------------+
|total members|
+-------------+
|           30|
+-------------+



In [0]:
# 6. Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID. group by multiple cols, join
result = (
    bookings.alias("b")
    .join(
        members.alias("m"),
        col("b.memid") == col("m.memid")
    )
    .filter(col("b.starttime") >= "2012-09-01")
    .groupBy("m.memid", "m.surname", "m.firstname")
    .agg(
        min("b.starttime").alias("first_booking")
    )
    .orderBy("m.memid")
)
result.show()

+-----+---------+---------+-------------------+
|memid|  surname|firstname|      first_booking|
+-----+---------+---------+-------------------+
|    0|    GUEST|    GUEST|2012-09-01 08:00:00|
|    1|    Smith|   Darren|2012-09-01 09:00:00|
|    2|    Smith|    Tracy|2012-09-01 11:30:00|
|    3|   Rownam|      Tim|2012-09-01 16:00:00|
|    4| Joplette|   Janice|2012-09-01 15:00:00|
|    5|  Butters|   Gerald|2012-09-02 12:30:00|
|    6|    Tracy|   Burton|2012-09-01 15:00:00|
|    7|     Dare|    Nancy|2012-09-01 12:30:00|
|    8|   Boothe|      Tim|2012-09-01 08:30:00|
|    9| Stibbons|   Ponder|2012-09-01 11:00:00|
|   10|     Owen|  Charles|2012-09-01 11:00:00|
|   11|    Jones|    David|2012-09-01 09:30:00|
|   12|    Baker|     Anne|2012-09-01 14:30:00|
|   13|  Farrell|   Jemima|2012-09-01 09:30:00|
|   14|    Smith|     Jack|2012-09-01 11:00:00|
|   15|    Bader| Florence|2012-09-01 10:30:00|
|   16|    Baker|  Timothy|2012-09-01 15:00:00|
|   17|   Pinker|    David|2012-09-01 08

### String & Date
1. https://pgexercises.com/questions/string/concat.html format string
2. https://pgexercises.com/questions/string/case.html WHERE + string function
3. https://pgexercises.com/questions/string/reg.html WHERE + string function
4. https://pgexercises.com/questions/string/substr.html group by, substr
5. https://pgexercises.com/questions/date/series.html generate ts
6. https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts

In [0]:
# 1. Output the names of all members, formatted as 'Surname, Firstname'
result = (
    members
    .select(concat_ws(", ", col("surname"), col("firstname")).alias("name"))
)
result.show()


+----------------+
|            name|
+----------------+
|    GUEST, GUEST|
|   Smith, Darren|
|    Smith, Tracy|
|     Rownam, Tim|
|Joplette, Janice|
| Butters, Gerald|
|   Tracy, Burton|
|     Dare, Nancy|
|     Boothe, Tim|
|Stibbons, Ponder|
|   Owen, Charles|
|    Jones, David|
|     Baker, Anne|
| Farrell, Jemima|
|     Smith, Jack|
| Bader, Florence|
|  Baker, Timothy|
|   Pinker, David|
|Genting, Matthew|
| Mackenzie, Anna|
+----------------+
only showing top 20 rows


In [0]:
# 2. Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns.
results = (
    facilities
    .filter(lower(col("name")).like("tennis%"))
)
results.show()


+-----+--------------+----------+---------+-------------+------------------+
|facid|          name|membercost|guestcost|initialoutlay|monthlymaintenance|
+-----+--------------+----------+---------+-------------+------------------+
|    0|Tennis Court 1|       5.0|     25.0|        10000|               200|
|    1|Tennis Court 2|       5.0|     25.0|         8000|               200|
+-----+--------------+----------+---------+-------------+------------------+



In [0]:
# 3. You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.
results = (
    members
    .filter(col("telephone").contains("("))
    .select("memid", "telephone")
    .orderBy("memid")
)
results.show()

+-----+--------------+
|memid|     telephone|
+-----+--------------+
|    0|(000) 000-0000|
|    3|(844) 693-0723|
|    4|(833) 942-4710|
|    5|(844) 078-4130|
|    6|(822) 354-9973|
|    7|(833) 776-4001|
|    8|(811) 433-2547|
|    9|(833) 160-3900|
|   10|(855) 542-5251|
|   11|(844) 536-8036|
|   13|(855) 016-0163|
|   14|(822) 163-3254|
|   15|(833) 499-3527|
|   20|(811) 972-1377|
|   21|(822) 661-2898|
|   22|(822) 499-2232|
|   24|(822) 413-1470|
|   27|(822) 989-8876|
|   28|(855) 755-9876|
|   29|(855) 894-3758|
+-----+--------------+
only showing top 20 rows


In [0]:
# 4. You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0.
results = (
    members
    .withColumn("letter", col("surname").substr(1, 1))
    .groupBy("letter")
    .agg(count("*").alias("count"))
    .orderBy("letter")
)
results.show()

+------+-----+
|letter|count|
+------+-----+
|     B|    5|
|     C|    2|
|     D|    1|
|     F|    2|
|     G|    2|
|     H|    1|
|     J|    3|
|     M|    1|
|     O|    1|
|     P|    2|
|     R|    2|
|     S|    6|
|     T|    2|
|     W|    1|
+------+-----+



In [0]:
# 5. Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.
result = (
    spark
    .range(1)
    .select(
        explode(
            sequence(
                to_date(lit("2012-10-01")),
                to_date(lit("2012-10-31"))
            )
        ).alias("date")
    )
)

result.show(31)

+----------+
|      date|
+----------+
|2012-10-01|
|2012-10-02|
|2012-10-03|
|2012-10-04|
|2012-10-05|
|2012-10-06|
|2012-10-07|
|2012-10-08|
|2012-10-09|
|2012-10-10|
|2012-10-11|
|2012-10-12|
|2012-10-13|
|2012-10-14|
|2012-10-15|
|2012-10-16|
|2012-10-17|
|2012-10-18|
|2012-10-19|
|2012-10-20|
|2012-10-21|
|2012-10-22|
|2012-10-23|
|2012-10-24|
|2012-10-25|
|2012-10-26|
|2012-10-27|
|2012-10-28|
|2012-10-29|
|2012-10-30|
|2012-10-31|
+----------+



In [0]:
# 6. Return a count of bookings for each month, sorted by month
results = (
    bookings
    .groupBy(month(col("starttime")).alias("month"))
    .agg(count("*").alias("count"))
    .orderBy("month")
)
results.show()

+-----+-----+
|month|count|
+-----+-----+
|    1|    1|
|    7|  658|
|    8| 1472|
|    9| 1913|
+-----+-----+

